In [ ]:
import customtkinter as ctk
import tkinter as tk
from tkinter import filedialog
import time
import threading

# --- Configuration & Theme ---
# Modes: "System" (standard), "Dark", "Light"
ctk.set_appearance_mode("Dark")  
# Themes: "blue" (standard), "green", "dark-blue"
ctk.set_default_color_theme("blue")  

class ModernReconTool(ctk.CTk):
    def __init__(self):
        super().__init__()

        # Window Setup
        self.title("Equipment Reconciliation Tool")
        self.geometry("600x650")
        self.resizable(False, False)

        # Data Storage
        self.source_path = tk.StringVar()
        self.master_path = tk.StringVar()
        self.export_path = tk.StringVar()

        # Layout Construction
        self.create_header()
        self.create_form()
        self.create_footer()

    def create_header(self):
        """Creates the title and subtitle section."""
        header_frame = ctk.CTkFrame(self, fg_color="transparent")
        header_frame.pack(pady=(40, 20), padx=40, fill="x")

        title = ctk.CTkLabel(
            header_frame, 
            text="Data Reconciliation", 
            font=("Roboto Medium", 28)
        )
        title.pack(anchor="w")

        subtitle = ctk.CTkLabel(
            header_frame, 
            text="UCSD Equipment & Part Number Mapping", 
            font=("Roboto", 14),
            text_color="gray"
        )
        subtitle.pack(anchor="w")

    def create_form(self):
        """Creates the main input card."""
        self.form_frame = ctk.CTkFrame(self, corner_radius=15)
        self.form_frame.pack(pady=10, padx=40, fill="both", expand=True)

        # --- Input 1: Source File ---
        self.create_file_input(
            parent=self.form_frame,
            label_text="Source File",
            desc_text="UCSD Equipment Summary (.xlsx, .csv)",
            variable=self.source_path,
            command=self.browse_source,
            icon="📄"
        )

        # Separator
        ctk.CTkFrame(self.form_frame, height=2, fg_color="#333333").pack(fill="x", padx=20, pady=5)

        # --- Input 2: Master File ---
        self.create_file_input(
            parent=self.form_frame,
            label_text="Master File",
            desc_text="Part Numbers Map Reference",
            variable=self.master_path,
            command=self.browse_master,
            icon="🗺️"
        )

        # Separator
        ctk.CTkFrame(self.form_frame, height=2, fg_color="#333333").pack(fill="x", padx=20, pady=5)

        # --- Input 3: Export Folder ---
        self.create_file_input(
            parent=self.form_frame,
            label_text="Export Location",
            desc_text="Where should the report be saved?",
            variable=self.export_path,
            command=self.browse_export,
            icon="📂"
        )

    def create_file_input(self, parent, label_text, desc_text, variable, command, icon):
        """Helper to create consistent input rows."""
        row_frame = ctk.CTkFrame(parent, fg_color="transparent")
        row_frame.pack(fill="x", padx=20, pady=15)

        # Text Section
        text_frame = ctk.CTkFrame(row_frame, fg_color="transparent")
        text_frame.pack(side="left", fill="x", expand=True)

        ctk.CTkLabel(
            text_frame, 
            text=f"{icon}  {label_text}", 
            font=("Roboto Medium", 14)
        ).pack(anchor="w")
        
        ctk.CTkLabel(
            text_frame, 
            text=desc_text, 
            font=("Roboto", 11), 
            text_color="gray"
        ).pack(anchor="w")

        # Entry Field (Read only display)
        entry = ctk.CTkEntry(
            row_frame, 
            textvariable=variable, 
            placeholder_text="No file selected",
            width=200,
            state="disabled", # Prevent typing, force button use
            fg_color="#1c1c1c",
            border_color="#333333"
        )
        entry.pack(side="right", padx=(10, 0))

        # Browse Button
        btn = ctk.CTkButton(
            row_frame, 
            text="Browse", 
            width=80, 
            height=30,
            fg_color="#2B2B2B",
            hover_color="#3A3A3A",
            command=command
        )
        btn.pack(side="right", padx=(10, 0))

    def create_footer(self):
        """Creates the action button and status bar."""
        footer_frame = ctk.CTkFrame(self, fg_color="transparent")
        footer_frame.pack(pady=20, padx=40, fill="x")

        # Progress Bar (Hidden initially)
        self.progress = ctk.CTkProgressBar(footer_frame, mode="indeterminate")
        self.progress.pack(fill="x", pady=(0, 15))
        self.progress.set(0)
        self.progress.pack_forget() # Hide it

        # Status Label
        self.status_label = ctk.CTkLabel(footer_frame, text="Ready to start.", text_color="gray")
        self.status_label.pack(pady=(0, 5))

        # Main Action Button
        self.run_btn = ctk.CTkButton(
            footer_frame,
            text="RUN RECONCILIATION",
            font=("Roboto Medium", 16),
            height=50,
            fg_color="#106A43", # A nicer green than the original
            hover_color="#148052",
            corner_radius=25,
            command=self.start_process
        )
        self.run_btn.pack(fill="x")

    # --- Logic Handlers ---

    def browse_source(self):
        filename = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx;*.xls"), ("CSV", "*.csv")])
        if filename:
            self.source_path.set(filename)

    def browse_master(self):
        filename = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx;*.xls")])
        if filename:
            self.master_path.set(filename)

    def browse_export(self):
        foldername = filedialog.askdirectory()
        if foldername:
            self.export_path.set(foldername)

    def start_process(self):
        """Simulates the backend processing."""
        # Validation
        if not all([self.source_path.get(), self.master_path.get(), self.export_path.get()]):
            self.status_label.configure(text="⚠️ Please select all files first!", text_color="#FF5555")
            return

        # UI State: Processing
        self.run_btn.configure(state="disabled", text="PROCESSING...")
        self.status_label.configure(text="Matching part numbers...", text_color="#3498db")
        self.progress.pack(fill="x", pady=(0, 15)) # Show progress
        self.progress.start()

        # Run in thread so UI doesn't freeze
        threading.Thread(target=self.run_reconciliation_logic).start()

    def run_reconciliation_logic(self):
        """Fake heavy lifting function."""
        time.sleep(1.5) # Simulate reading files
        self.status_label.configure(text="Generating report...")
        time.sleep(1.5) # Simulate writing export

        # Reset UI
        self.progress.stop()
        self.progress.pack_forget()
        self.run_btn.configure(state="normal", text="RUN RECONCILIATION")
        self.status_label.configure(text="✅ Reconciliation Complete!", text_color="#2cc985")
        print("Done.")

if __name__ == "__main__":
    app = ModernReconTool()
    app.mainloop()

ModuleNotFoundError: No module named 'customtkinter'